In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

In [2]:
# problem description

# We are given some forecasts for demands for our products.
# We want to optimise how much of each product to store at any given point in time 
# in order to maximise profit (maximum revenue and minimum cost).
# We only have limited amount of storage for each product category ()
# and each product carries a penalty cost for storing excess product in the warehouse.

In [12]:
# Constants

INPUT_BASE_PATH = "../data"
FORECASTS_PATH = Path("../results/ets/submissions")

# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1941
FORECAST_HORIZON = 28
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

# ==================================================
# OPTIMIZATION CONSTANTS / PARAMETERS / CONSTRAINTS
# ==================================================
TOTAL_STORAGE_CAPACITY = 45_000

# Gamma distributed
FOOD_VOLUME_DIST_PARAMS = {"shape": 5, "scale": 10}
HOUSEHOLD_VOLUME_DIST_PARAMS = {"shape": 10, "scale": 15}
HOBBIES_VOLUME_DIST_PARAMS = {"shape": 15, "scale": 17}

# Uniformly distributed
FOOD_STORAGE_COST_DIST_PARAMS = {"low": 0.1, "high": 0.5}
HOUSEHOLD_STORAGE_COST_DIST_PARAMS = {"low": 0.2, "high": 0.3}
HOBBIES_STORAGE_COST_DIST_PARAMS = {"low": 0.3, "high": 0.7}

PRODUCT_ID_SUFFIX = "_evaluation"
FOODS_PRODUCT_IDS = [
    "FOODS_3_555_TX_2",
    "FOODS_3_376_TX_2",
    "FOODS_3_811_CA_2",
    "FOODS_1_218_TX_1",
    "FOODS_3_226_WI_3",
    "FOODS_3_070_WI_2",
    "FOODS_3_007_TX_2",
    "FOODS_3_444_TX_1",
    "FOODS_2_398_WI_3",
    "FOODS_3_540_WI_1",
]
HOUSEHOLD_PRODUCT_IDS = [
    "HOUSEHOLD_1_334_TX_1",
    "HOUSEHOLD_1_459_CA_2",
    "HOUSEHOLD_2_342_WI_2",
    "HOUSEHOLD_1_465_TX_3",
    "HOUSEHOLD_1_294_WI_2",
    "HOUSEHOLD_2_176_CA_3",
    "HOUSEHOLD_1_334_TX_2",
    "HOUSEHOLD_1_106_WI_2",
    "HOUSEHOLD_1_474_TX_2",
    "HOUSEHOLD_1_106_TX_1",
]
HOBBIES_PRODUCT_IDS = [
    "HOBBIES_1_048_WI_1",
    "HOBBIES_1_067_CA_3",
    "HOBBIES_1_158_TX_3",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_234_CA_3",
    "HOBBIES_1_254_CA_3",
    "HOBBIES_1_019_WI_1",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]
ALL_PRODUCT_IDS = list(FOODS_PRODUCT_IDS + HOUSEHOLD_PRODUCT_IDS + HOBBIES_PRODUCT_IDS)
ALL_PRODUCT_IDS_WITH_SUFFIX = [f"{p}{PRODUCT_ID_SUFFIX}" for p in ALL_PRODUCT_IDS]

RNG = np.random.default_rng(42)

In [4]:
# Load data and forecasts

SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
FORECASTS = pl.read_csv(f"{FORECASTS_PATH}/submission_2026-07-12 21:48:22.csv")

In [5]:
# Prepare forecast df

forecast_df = (
    FORECASTS.filter(pl.col("id").is_in(ALL_PRODUCT_IDS_WITH_SUFFIX))
    .unpivot(index="id", value_name="sales", variable_name="F")
    .with_columns(
        item_store_id=pl.col("id").str.strip_suffix(PRODUCT_ID_SUFFIX),
        dept_id=pl.col("id").str.extract(r"^([^_]+)"),
        F_index=pl.col("F").str.strip_chars_start("F").cast(pl.Int64)
    )
    .with_columns(
        d=pl.format("d_{}", pl.col("F_index") + MAX_TRAINING_TIMESTAMP),
        d_index=pl.col("F_index") + MAX_TRAINING_TIMESTAMP
    )
    .select(["id", "item_store_id", "dept_id", "F", "F_index", "d", "d_index", "sales"])
)

forecast_df

id,item_store_id,dept_id,F,F_index,d,d_index,sales
str,str,str,str,i64,str,i64,i64
"""HOBBIES_1_048_CA_1_evaluation""","""HOBBIES_1_048_CA_1""","""HOBBIES""","""F1""",1,"""d_1942""",1942,15
"""HOUSEHOLD_1_459_CA_2_evaluatio…","""HOUSEHOLD_1_459_CA_2""","""HOUSEHOLD""","""F1""",1,"""d_1942""",1942,9
"""FOODS_3_811_CA_2_evaluation""","""FOODS_3_811_CA_2""","""FOODS""","""F1""",1,"""d_1942""",1942,10
"""HOBBIES_1_067_CA_3_evaluation""","""HOBBIES_1_067_CA_3""","""HOBBIES""","""F1""",1,"""d_1942""",1942,4
"""HOBBIES_1_234_CA_3_evaluation""","""HOBBIES_1_234_CA_3""","""HOBBIES""","""F1""",1,"""d_1942""",1942,22
…,…,…,…,…,…,…,…
"""HOUSEHOLD_2_342_WI_2_evaluatio…","""HOUSEHOLD_2_342_WI_2""","""HOUSEHOLD""","""F28""",28,"""d_1969""",1969,13
"""FOODS_3_070_WI_2_evaluation""","""FOODS_3_070_WI_2""","""FOODS""","""F28""",28,"""d_1969""",1969,13
"""HOBBIES_1_404_WI_3_evaluation""","""HOBBIES_1_404_WI_3""","""HOBBIES""","""F28""",28,"""d_1969""",1969,3


In [6]:
# Join calendar and price data onto forecasts

full_df = (
    forecast_df
    .join(
        CALENDAR_DATA.select(["d", "date", "wm_yr_wk"]),
        how="left",
        on="d"
    )
    .join(
        SELL_PRICES.with_columns(item_store_id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id"))),
        how="left",
        on=["item_store_id", "wm_yr_wk"]
    )
    .select(["id", "item_id", "store_id", "item_store_id", "dept_id", "F", "F_index", "d", "d_index", "date", "wm_yr_wk", "sales", "sell_price"])
)

full_df.head()

id,item_id,store_id,item_store_id,dept_id,F,F_index,d,d_index,date,wm_yr_wk,sales,sell_price
str,str,str,str,str,str,i64,str,i64,date,i64,i64,f64
"""HOBBIES_1_048_CA_1_evaluation""","""HOBBIES_1_048""","""CA_1""","""HOBBIES_1_048_CA_1""","""HOBBIES""","""F1""",1,"""d_1942""",1942,2016-05-23,11617,15,0.48
"""HOUSEHOLD_1_459_CA_2_evaluatio…","""HOUSEHOLD_1_459""","""CA_2""","""HOUSEHOLD_1_459_CA_2""","""HOUSEHOLD""","""F1""",1,"""d_1942""",1942,2016-05-23,11617,9,0.97
"""FOODS_3_811_CA_2_evaluation""","""FOODS_3_811""","""CA_2""","""FOODS_3_811_CA_2""","""FOODS""","""F1""",1,"""d_1942""",1942,2016-05-23,11617,10,1.88
"""HOBBIES_1_067_CA_3_evaluation""","""HOBBIES_1_067""","""CA_3""","""HOBBIES_1_067_CA_3""","""HOBBIES""","""F1""",1,"""d_1942""",1942,2016-05-23,11617,4,0.57
"""HOBBIES_1_234_CA_3_evaluation""","""HOBBIES_1_234""","""CA_3""","""HOBBIES_1_234_CA_3""","""HOBBIES""","""F1""",1,"""d_1942""",1942,2016-05-23,11617,22,0.3


In [46]:
# Define storage volume per product
food_product_volumes = RNG.gamma(**FOOD_VOLUME_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_volume = pl.DataFrame({"item_store_id": FOODS_PRODUCT_IDS, "volume": food_product_volumes})

household_product_volumes = RNG.gamma(**HOUSEHOLD_VOLUME_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_volume = pl.DataFrame({"item_store_id": HOUSEHOLD_PRODUCT_IDS, "volume": household_product_volumes})

hobbies_product_volumes = RNG.gamma(**HOBBIES_VOLUME_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_volume = pl.DataFrame({"item_store_id": HOBBIES_PRODUCT_IDS, "volume": hobbies_product_volumes})

VOLUME_DF = (
    full_df.select(["id", "item_id", "item_store_id"])
    .join(
        other=food_product_id_to_volume.rename({"volume": "volume_foods"}),
        on="item_store_id",
        how="left"
    ).join(
        other=household_product_id_to_volume.rename({"volume": "volume_household"}),
        on="item_store_id",
        how="left",
    ).join(
        other=hobbies_product_id_to_volume.rename({"volume": "volume_hobbies"}),
        on="item_store_id",
        how="left"
    )
    .with_columns(
        storage_volume=pl.coalesce(pl.selectors.starts_with("volume_"))
    ).drop(
        pl.selectors.starts_with("volume_")
    )
)

VOLUME_DF

id,item_id,item_store_id,storage_volume
str,str,str,f64
"""HOBBIES_1_048_CA_1_evaluation""","""HOBBIES_1_048""","""HOBBIES_1_048_CA_1""",196.491445
"""HOUSEHOLD_1_459_CA_2_evaluatio…","""HOUSEHOLD_1_459""","""HOUSEHOLD_1_459_CA_2""",192.046528
"""FOODS_3_811_CA_2_evaluation""","""FOODS_3_811""","""FOODS_3_811_CA_2""",38.86867
"""HOBBIES_1_067_CA_3_evaluation""","""HOBBIES_1_067""","""HOBBIES_1_067_CA_3""",197.806232
"""HOBBIES_1_234_CA_3_evaluation""","""HOBBIES_1_234""","""HOBBIES_1_234_CA_3""",204.814774
…,…,…,…
"""HOUSEHOLD_2_342_WI_2_evaluatio…","""HOUSEHOLD_2_342""","""HOUSEHOLD_2_342_WI_2""",146.433254
"""FOODS_3_070_WI_2_evaluation""","""FOODS_3_070""","""FOODS_3_070_WI_2""",46.556109
"""HOBBIES_1_404_WI_3_evaluation""","""HOBBIES_1_404""","""HOBBIES_1_404_WI_3""",270.450416


In [ ]:
# Define storage cost per item as a proportion of average sell
# price for item.

average_sell_prices = (
    full_df
    .group_by(pl.col("item_store_id"), pl.col("dept_id"))
    .agg(pl.col("sell_price").mean())
    .sort(pl.col("dept_id"), pl.col("item_store_id"))
)

food_storage_costs_pct = RNG.uniform(**FOOD_STORAGE_COST_DIST_PARAMS, size=len(FOODS_PRODUCT_IDS))
food_product_id_to_cost_pct = pl.DataFrame({"item_store_id": FOODS_PRODUCT_IDS, "cost_pct": food_storage_costs_pct})

household_storage_costs_pct = RNG.uniform(**HOUSEHOLD_STORAGE_COST_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_IDS))
household_product_id_to_cost_pct = pl.DataFrame({"item_store_id": HOUSEHOLD_PRODUCT_IDS, "cost_pct": household_storage_costs_pct})

hobbies_storage_costs_pct = RNG.uniform(**HOBBIES_STORAGE_COST_DIST_PARAMS, size=len(HOBBIES_PRODUCT_IDS))
hobbies_product_id_to_cost_pct = pl.DataFrame({"item_store_id": HOBBIES_PRODUCT_IDS, "cost_pct": hobbies_storage_costs_pct})

PRICE_COST_DF = (
    average_sell_prices.join(
        other=food_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_food"}),
        how="left",
        on="item_store_id",
    ).join(
        other=household_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_household"}),
        how="left",
        on="item_store_id"
    ).join(
        other=hobbies_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_hobbies"}),
        how="left",
        on="item_store_id"
    ).with_columns(
        cost_pct=pl.coalesce(pl.selectors.starts_with("cost_pct_"))
    ).with_columns(
        storage_cost=(pl.col("cost_pct") * pl.col("sell_price")).round(2)
    ).drop(
        pl.selectors.starts_with("cost_pct")
    )
)

In [ ]:
# Define total storage cost limit.

